# 00 · PreparedReviewReader interface lab

This notebook is a manual debugging surface for Group A's real data interface. It calls only `yelptext.interface.data_corpus_analysis`; it does not duplicate reader implementation.

The default configuration uses the small synthetic fixture. Change `CONFIG_PATH` to `configs/default.toml` only when the real Yelp files are available. All inspection is bounded by explicit limits, so the full review file is never loaded into a dataframe.

In [ ]:
from __future__ import annotations

from collections import Counter
from html import escape
from itertools import islice
from pathlib import Path
from typing import get_type_hints
import json
import sys

from IPython.display import HTML, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError('Run this notebook from inside the repository.')


def show_table(rows: list[dict], columns: list[str] | None = None) -> None:
    if not rows:
        display(HTML('<em>No rows to display.</em>'))
        return
    columns = columns or list(rows[0])

    def format_value(value: object) -> str:
        if isinstance(value, (list, dict)):
            value = json.dumps(value, ensure_ascii=False)
        return escape(str(value))

    header = ''.join(f'<th>{escape(column)}</th>' for column in columns)
    body = ''.join(
        '<tr>' + ''.join(f'<td>{format_value(row.get(column))}</td>' for column in columns) + '</tr>'
        for row in rows
    )
    display(HTML(
        '<div style="overflow-x:auto"><table style="border-collapse:collapse">'
        f'<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>'
    ))


ROOT = find_repo_root(Path.cwd().resolve())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from yelptext.interface.data_corpus_analysis import (
    YelpDataError,
    create_review_reader,
)
from yelptext.protocol.reviewer import PreparedReview
from yelptext.runtime import load_config

print(f'Repository: {ROOT}')

## Controls

Edit this cell and rerun the cells below. `skip` is convenient for browsing the fixture; `error` is the strict integration behavior.

In [ ]:
CONFIG_PATH = ROOT / 'tests/fixtures/fixture.toml'
# CONFIG_PATH = ROOT / 'configs/default.toml'

SAMPLE_LIMIT = 20
ORPHAN_POLICY = 'skip'  # 'skip' or 'error'
TEXT_PREVIEW_CHARS = 180

# Optional manual browser filters. Use None to disable a filter.
CATEGORY_FILTER: str | None = None
TEXT_FILTER: str | None = None
MAX_RECORDS_TO_SCAN = 500
MATCH_LIMIT = 10

In [ ]:
config = load_config(CONFIG_PATH)
data_config = config['data']


def repo_path(value: str) -> Path:
    path = Path(value)
    return path if path.is_absolute() else ROOT / path


review_path = repo_path(data_config['review_path'])
business_path = repo_path(data_config['business_path'])
reader = create_review_reader(
    review_path,
    business_path,
    orphan_policy=ORPHAN_POLICY,
)

source_status = [
    {'source': 'reviews', 'path': str(review_path), 'exists': review_path.is_file(), 'bytes': review_path.stat().st_size if review_path.is_file() else None},
    {'source': 'businesses', 'path': str(business_path), 'exists': business_path.is_file(), 'bytes': business_path.stat().st_size if business_path.is_file() else None},
]
show_table(source_status)
print(f'Interface object: {type(reader).__module__}.{type(reader).__name__}')

## Bounded sample

The table shows exactly what B/C receive. `text_preview` is presentation-only; the interface record still contains the complete `text`.

In [ ]:
records = list(islice(reader.iter_reviews(), SAMPLE_LIMIT))
sample_rows = [
    {
        **{key: value for key, value in record.items() if key != 'text'},
        'text_preview': record['text'][:TEXT_PREVIEW_CHARS],
    }
    for record in records
]
show_table(sample_rows)
print(f'Records loaded: {len(records)} (limit={SAMPLE_LIMIT})')

## Protocol conformance

This checks exact field names and runtime value types. It is a debugging aid, not a replacement for the automated tests.

In [ ]:
type_hints = get_type_hints(PreparedReview)
expected_fields = tuple(type_hints)
runtime_types = {
    'review_id': str,
    'business_id': str,
    'business_name': str,
    'categories': list,
    'stars': float,
    'text': str,
    'date': str,
}

schema_rows = []
for field in expected_fields:
    missing = sum(field not in record for record in records)
    wrong_type = sum(
        field in record and not isinstance(record[field], runtime_types[field])
        for record in records
    )
    schema_rows.append({
        'field': field,
        'declared_type': str(type_hints[field]),
        'missing': missing,
        'wrong_runtime_type': wrong_type,
    })

unexpected = sorted({key for record in records for key in record} - set(expected_fields))
field_order_matches = all(tuple(record) == expected_fields for record in records)
show_table(schema_rows)
print(f'Expected field order: {expected_fields}')
print(f'All records use that exact order: {field_order_matches}')
unexpected_display = unexpected if unexpected else 'none'
print(f'Unexpected fields: {unexpected_display}')

## Sample diagnostics

Counts below apply only to the bounded sample. They help assess whether the interface carries enough business metadata for search display and category/sector trend analysis.

In [ ]:
if records:
    category_counts = Counter(
        category
        for record in records
        for category in record['categories']
    )
    summary = {
        'sample_records': len(records),
        'unique_review_ids': len({record['review_id'] for record in records}),
        'unique_business_ids': len({record['business_id'] for record in records}),
        'earliest_date': min(record['date'] for record in records),
        'latest_date': max(record['date'] for record in records),
        'mean_stars': sum(record['stars'] for record in records) / len(records),
        'records_without_categories': sum(not record['categories'] for record in records),
    }
    show_table([{'metric': key, 'value': value} for key, value in summary.items()])
    show_table([
        {'category': category, 'sample_count': count}
        for category, count in category_counts.most_common(15)
    ])

## Manual record browser

Set `CATEGORY_FILTER` or `TEXT_FILTER` in the controls cell. Matching is case-insensitive and scanning stops at `MAX_RECORDS_TO_SCAN`.

In [ ]:
def matches(record: PreparedReview) -> bool:
    if CATEGORY_FILTER is not None and not any(
        CATEGORY_FILTER.casefold() in category.casefold()
        for category in record['categories']
    ):
        return False
    if TEXT_FILTER is not None and TEXT_FILTER.casefold() not in record['text'].casefold():
        return False
    return True


matches_found = []
for record in islice(reader.iter_reviews(), MAX_RECORDS_TO_SCAN):
    if matches(record):
        matches_found.append(record)
        if len(matches_found) >= MATCH_LIMIT:
            break

browser_rows = [
    {
        **{key: value for key, value in record.items() if key != 'text'},
        'text_preview': record['text'][:TEXT_PREVIEW_CHARS],
    }
    for record in matches_found
]
show_table(browser_rows)
print(f'Matches shown: {len(matches_found)}')

## Strict orphan-policy probe

The synthetic fixture deliberately contains one review whose business is missing. With the strict policy, the interface should report the source file, line, and missing business ID. Keep the scan bounded when using real data.

In [ ]:
RUN_STRICT_PROBE = True
STRICT_SCAN_LIMIT = 50

if RUN_STRICT_PROBE:
    strict_reader = create_review_reader(
        review_path,
        business_path,
        orphan_policy='error',
    )
    try:
        strict_count = sum(1 for _ in islice(strict_reader.iter_reviews(), STRICT_SCAN_LIMIT))
        print(f'No orphan encountered in the first {strict_count} records.')
    except YelpDataError as exc:
        print(f'Expected strict-reader diagnostic: {exc}')

## Manual review checklist

- Are the seven public fields sufficient for B and C?
- Is `YYYY-MM-DD` enough for time aggregation, or is time-of-day needed?
- Are `business_name` and `categories` useful enough to remain in every record?
- Should an orphan stop a production run or be counted and skipped?
- Are null Yelp categories correctly represented as `[]`?
- Does the real-data sample expose encoding or category-normalization problems?

Record decisions in the protocol before B/C generate their final fake inputs.